# 04. Database Connection and Table Creation

## Objective

The objective of this notebook is to establish a connection between Python and MySQL, create the database required for the PhonePe Transaction Insights project, and define all relational tables needed for storing structured transaction, user, insurance, map, and top-level data. This stage is an important part of the project pipeline because all further SQL analysis, Python visualization, and dashboard development depend on a properly designed and validated database schema.

## Why this notebook is important

The raw PhonePe Pulse data is available in nested folder and JSON structure, which is not suitable for direct SQL analytics. To support structured querying and business analysis, the transformed data must be stored in a relational database with properly defined tables. In this notebook, the schema is organized into three major groups: Aggregated tables, Map tables, and Top tables.

## Import Libraries

In [1]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## Database Credentials

Enter your local MySQL credentials below. Make sure the MySQL server is already running before executing the connection cells.

In [2]:
# MySQL connection details
HOST = "localhost"
USER = "root"
PASSWORD = "12345"
DATABASE = "phonepe_insights"

## Create Connection to MySQL Server

In [3]:
# Connect to MySQL server without selecting database first
try:
    server_connection = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD
    )
    
    if server_connection.is_connected():
        print("Successfully connected to MySQL Server.")
        
except Error as e:
    print("Error while connecting to MySQL Server:", e)

Successfully connected to MySQL Server.


## Create Project Database

In [4]:
# Create database if it does not exist
try:
    cursor = server_connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
    print(f"Database '{DATABASE}' created successfully or already exists.")
except Error as e:
    print("Error while creating database:", e)

Database 'phonepe_insights' created successfully or already exists.


## Connect to Project Database

In [5]:
# Connect to the created database
try:
    db_connection = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        database=DATABASE
    )
    
    if db_connection.is_connected():
        print(f"Successfully connected to database: {DATABASE}")
        db_cursor = db_connection.cursor()
        
except Error as e:
    print("Error while connecting to project database:", e)

Successfully connected to database: phonepe_insights


## SQLAlchemy Engine Creation

SQLAlchemy is used to simplify interaction between Python DataFrames and MySQL tables. This engine will also be used later during data insertion and SQL-based retrieval for analysis.

In [6]:
# SQLAlchemy engine
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}")
print("SQLAlchemy engine created successfully.")

SQLAlchemy engine created successfully.


## Table Design Overview

The table structure is divided into the following categories:

1. Aggregated Tables  
   - aggregated_user  
   - aggregated_transaction  
   - aggregated_insurance  

2. Map Tables  
   - map_user  
   - map_transaction  
   - map_insurance  

3. Top Tables  
   - top_user  
   - top_transaction  
   - top_insurance  

This structure is based on the project design shared in the PhonePe project documentation.

## Create Aggregated Tables

In [7]:
create_aggregated_user = """
CREATE TABLE IF NOT EXISTS aggregated_user (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    brand VARCHAR(100),
    count BIGINT,
    percentage FLOAT
);
"""

create_aggregated_transaction = """
CREATE TABLE IF NOT EXISTS aggregated_transaction (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    transaction_type VARCHAR(150),
    transaction_count BIGINT,
    transaction_amount DOUBLE
);
"""

create_aggregated_insurance = """
CREATE TABLE IF NOT EXISTS aggregated_insurance (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    insurance_type VARCHAR(150),
    insurance_count BIGINT,
    insurance_amount DOUBLE
);
"""

In [8]:
try:
    db_cursor.execute(create_aggregated_user)
    db_cursor.execute(create_aggregated_transaction)
    db_cursor.execute(create_aggregated_insurance)
    db_connection.commit()
    print("Aggregated tables created successfully.")
except Error as e:
    print("Error while creating aggregated tables:", e)

Aggregated tables created successfully.


## Create Map Tables

In [9]:
create_map_user = """
CREATE TABLE IF NOT EXISTS map_user (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    district VARCHAR(150),
    registered_users BIGINT,
    app_opens BIGINT
);
"""

create_map_transaction = """
CREATE TABLE IF NOT EXISTS map_transaction (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    district VARCHAR(150),
    transaction_count BIGINT,
    transaction_amount DOUBLE
);
"""

create_map_insurance = """
CREATE TABLE IF NOT EXISTS map_insurance (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    district VARCHAR(150),
    insurance_count BIGINT,
    insurance_amount DOUBLE
);
"""

In [10]:
try:
    db_cursor.execute(create_map_user)
    db_cursor.execute(create_map_transaction)
    db_cursor.execute(create_map_insurance)
    db_connection.commit()
    print("Map tables created successfully.")
except Error as e:
    print("Error while creating map tables:", e)

Map tables created successfully.


## Create Top Tables

In [11]:
create_top_user = """
CREATE TABLE IF NOT EXISTS top_user (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    district VARCHAR(150),
    registered_users BIGINT,
    pincode BIGINT
);
"""

create_top_transaction = """
CREATE TABLE IF NOT EXISTS top_transaction (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    entity_name VARCHAR(150),
    entity_type VARCHAR(50),
    transaction_count BIGINT,
    transaction_amount DOUBLE,
    pincode BIGINT
);
"""

create_top_insurance = """
CREATE TABLE IF NOT EXISTS top_insurance (
    id INT AUTO_INCREMENT PRIMARY KEY,
    state VARCHAR(100),
    year INT,
    quarter INT,
    entity_name VARCHAR(150),
    entity_type VARCHAR(50),
    insurance_count BIGINT,
    insurance_amount DOUBLE,
    pincode BIGINT
);
"""

In [12]:
try:
    db_cursor.execute(create_top_user)
    db_cursor.execute(create_top_transaction)
    db_cursor.execute(create_top_insurance)
    db_connection.commit()
    print("Top tables created successfully.")
except Error as e:
    print("Error while creating top tables:", e)

Top tables created successfully.


## Verify All Created Tables

In [13]:
# Show all tables in the database
try:
    db_cursor.execute("SHOW TABLES;")
    tables = db_cursor.fetchall()
    print("Tables available in the database:\n")
    for table in tables:
        print(table[0])
except Error as e:
    print("Error while fetching table list:", e)

Tables available in the database:

aggregated_insurance
aggregated_transaction
aggregated_user
map_insurance
map_transaction
map_user
top_insurance
top_transaction
top_user


## Preview Table Schema

In [14]:
# Check schema of one or more tables
table_names = [
    "aggregated_user",
    "aggregated_transaction",
    "aggregated_insurance",
    "map_user",
    "map_transaction",
    "map_insurance",
    "top_user",
    "top_transaction",
    "top_insurance"
]

for table in table_names:
    print(f"\nSchema for table: {table}")
    try:
        db_cursor.execute(f"DESCRIBE {table};")
        schema_info = db_cursor.fetchall()
        schema_df = pd.DataFrame(schema_info, columns=["Field", "Type", "Null", "Key", "Default", "Extra"])
        display(schema_df)
    except Error as e:
        print(f"Error while describing table {table}: {e}")


Schema for table: aggregated_user


,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,state,varchar(100),YES,,None,
2,year,int,YES,,None,
3,quarter,int,YES,,None,
4,brand,varchar(100),YES,,None,
5,count,bigint,YES,,None,
6,percentage,double,YES,,None,



Schema for table: aggregated_transaction


,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,state,varchar(100),YES,,None,
2,year,int,YES,,None,
3,quarter,int,YES,,None,
4,transaction_type,varchar(100),YES,,None,
5,transaction_count,bigint,YES,,None,
6,transaction_amount,double,YES,,None,



Schema for table: aggregated_insurance


...

,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,state,varchar(100),YES,,None,
2,year,int,YES,,None,
3,quarter,int,YES,,None,
4,entity_type,varchar(50),YES,,None,
5,entity_name,varchar(150),YES,,None,
6,registered_users,bigint,YES,,None,



Schema for table: top_transaction


,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,state,varchar(100),YES,,None,
2,year,int,YES,,None,
3,quarter,int,YES,,None,
4,entity_type,varchar(50),YES,,None,
5,entity_name,varchar(150),YES,,None,
6,transaction_count,bigint,YES,,None,
7,transaction_amount,double,YES,,None,



Schema for table: top_insurance


,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,state,varchar(100),YES,,None,
2,year,int,YES,,None,
3,quarter,int,YES,,None,
4,entity_name,varchar(150),YES,,None,
5,entity_type,varchar(50),YES,,None,
6,insurance_count,bigint,YES,,None,
7,insurance_amount,double,YES,,None,
8,pincode,bigint,YES,,None,


## What did you know about your database schema?

The relational schema for the PhonePe Transaction Insights project was successfully created in MySQL. The database design was organized into three logical groups—aggregated, map, and top tables—so that different layers of PhonePe data can be stored in a structured and query-friendly format. This structure makes it easier to run SQL business queries for state-level, district-level, category-level, and pincode-level analysis.

The aggregated tables are intended to store high-level transaction, user, and insurance summaries. The map tables are designed to preserve district-wise activity and geographic metrics. The top tables support ranking-based analysis by storing top states, districts, or pin code entities. This schema is suitable for the next project phase, where transformed data will be inserted into the database and validated before analytics.

## Best Practices Followed

1. The database was created separately before table creation to avoid connection issues.
2. The schema was divided according to the project’s logical data groups.
3. Data types such as BIGINT and DOUBLE were used for high-volume transaction fields.
4. SQLAlchemy engine support was added for easier future insertion using Pandas.
5. Validation was performed using both `SHOW TABLES` and `DESCRIBE` queries.

## Conclusion

In this notebook, the MySQL connection was established successfully, the PhonePe project database was created, and all required tables were defined using a structured schema. The database is now ready for the next phase of the project, which is loading cleaned and transformed PhonePe data into the SQL tables for analytical querying and dashboard preparation.

In [15]:
# Optional: close cursor and keep connection active only if needed
# db_cursor.close()
# db_connection.close()
# server_connection.close()